In [55]:
# Setup
import pandas as pd
import numpy as np
from pathlib import Path

# Define path to data folder
DATA_DIR = Path.cwd().parents[2] / 'data' / 'spreadsheets'

# Read in Population Data
pop2020 = pd.read_excel(DATA_DIR / 'nga_admpop_2020.xlsx', sheet_name='nga_admpop_adm2_2020')

# Read in COD-PS Nigeria Intrinsic Population Growth Rate Data
cod = pd.read_excel(DATA_DIR / 'NGA_PGR_2022.xlsx', sheet_name='PGR_Proj')
cod = cod.rename(columns={
    'PGR_BOTH_2022': 'ANN_POP_CHANGE'
})

# Read in City Population Annual Population Change (2006-2022) data
city = pd.read_excel(DATA_DIR / 'NGA_citypop_apc.xlsx', sheet_name='cp_apc')

# Merge COD-PS Intrinsic Population Growth Rate Data with City Population Annual Population Change Rate Data for comparison
pop_growth_merged = city.merge(cod, on='ADM1_NAME', how='outer', suffixes=('_city', '_cod'))
print(pop_growth_merged.head())


   ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod
0       Abia                  2.4                2.35
1    Adamawa                  2.7                2.71
2  Akwa Ibom                  1.5                1.52
3    Anambra                  2.2                2.21
4     Bauchi                  3.7                3.62


In [74]:
# Identify missing states in COD-PS data vs City Population data
missing_in_cod = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_cod'].isna()][['ADM1_NAME', 'ANN_POP_CHANGE_city']]
print("States missing from COD-PS:\n", missing_in_cod)

missing_in_city = pop_growth_merged[pop_growth_merged['ANN_POP_CHANGE_city'].isna()][['ADM1_NAME', 'ANN_POP_CHANGE_cod']]
print("\nStates missing from City Population data:\n", missing_in_city)

States missing from COD-PS:
    ADM1_NAME  ANN_POP_CHANGE_city
25  Nasarawa                  2.8

States missing from City Population data:
 Empty DataFrame
Columns: [ADM1_NAME, ANN_POP_CHANGE_cod]
Index: []


Nasarawa is missing data in COD-PS. We will measure sameness/difference in the data next to determine if we can fill in missing COD-PS data with City Population data.

In [75]:
# Measure sameness/difference in rates data

# Work with states that have data points in both tables
both = pop_growth_merged.dropna(subset=['ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod']).copy()

# Absolute Difference
both['abs_diff'] = (both['ANN_POP_CHANGE_city'] - both['ANN_POP_CHANGE_cod']).abs()

# Check if they match within a rounding tolerance
both['within_rounding_tolerance'] = both['abs_diff'] <= 0.1

# Summary metrics
print("Max absolute difference:   ", both['abs_diff'].max())
print("Mean absolute difference:  ", both['abs_diff'].mean())
print("States within rounding:    ", both['within_rounding_tolerance'].sum(), "of", len(both))
print("Correlation:               ", both['ANN_POP_CHANGE_city'].corr(both['ANN_POP_CHANGE_cod']).round(4))

# See any states that diverge beyond rounding
outliers = both[~both['within_rounding_tolerance']]
print("\nStates outside rounding tolerance:\n", outliers[['ADM1_NAME', 'ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod', 'abs_diff']])

Max absolute difference:    5.2
Mean absolute difference:   0.19194444444444442
States within rounding:     33 of 36
Correlation:                0.7154

States outside rounding tolerance:
                     ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod  \
14  Federal Capital Territory                  5.0                4.88   
17                     Jigawa                  3.5                3.40   
36                    Zamfara                  3.7                8.90   

    abs_diff  
14      0.12  
17      0.10  
36      5.20  


The annual population change data between the two datasets appears to be the same. We will fill in the missing Nasarawa COD-PS rate with the City Population rate.

In our outlier check, FCT (Abuja), Jigawa, and Zamfara are flagged as outliers. While FCT and Jigawa look like rounding errors (5.0 vs 4.88 and 3.5 vs 3.4 respectively in City Population vs COD-PS data), Zamfara looks like completely different rates (3.7 vs 8.9 in City Population vs COD-PS data). We will trust the City Population data. This will potentially underestimate our population growth.

In [85]:
# Decision: fill missing COD-PS rate for Nasarawa with City Population rates
#pop_growth_merged['ANN_POP_CHANGE_cod'] = pop_growth_merged['ANN_POP_CHANGE_cod'].fillna(pop_growth_merged['ANN_POP_CHANGE_city'])
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_cod'] = pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Nasarawa', 'ANN_POP_CHANGE_city']

# Flag Zamfara discrepancy
zamfara = pop_growth_merged[pop_growth_merged['ADM1_NAME'] == 'Zamfara']
print(zamfara[['ADM1_NAME', 'ANN_POP_CHANGE_city', 'ANN_POP_CHANGE_cod']])

# Option A: Trust City Population
pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Zamfara', 'rate_final'] = 3.7

# Option B: Trust COD-PS
#pop_growth_merged.loc[pop_growth_merged['ADM1_NAME'] == 'Zamfara', 'rate_final'] = 8.9

# Final Reconciled Column
pop_growth_merged['rate_final'] = pop_growth_merged['rate_final'].fillna(pop_growth_merged['ANN_POP_CHANGE_cod'])

# Sanity check - no nulls
print("\nFinal population growth rates:\n", pop_growth_merged[['ADM1_NAME', 'rate_final']])

   ADM1_NAME  ANN_POP_CHANGE_city  ANN_POP_CHANGE_cod
36   Zamfara                  3.7                 8.9

Final population growth rates:
                     ADM1_NAME  rate_final
0                        Abia        2.35
1                     Adamawa        2.71
2                   Akwa Ibom        1.52
3                     Anambra        2.21
4                      Bauchi        3.62
5                     Bayelsa        2.49
6                       Benue        2.30
7                       Borno        2.39
8                 Cross River        2.63
9                       Delta        1.97
10                     Ebonyi        2.49
11                        Edo        2.44
12                      Ekiti        2.53
13                      Enugu        2.26
14  Federal Capital Territory        4.88
15                      Gombe        3.22
16                        Imo        2.06
17                     Jigawa        3.40
18                     Kaduna        2.44
19                 

In [86]:


print(pop.head(10))

print(pop['SenDist_en'].nunique())
print(pop['ADM2_NAME'].nunique())

# Rename T_TL to POP_2020 for clarity
pop.rename(columns={'T_TL': 'POP_2020'}, inplace=True)



# Drop PGR_PROJ_2025 column from pgr dataframe since we will calculate it ourselves
pgr.drop(columns=['PGR_PROJ_2025'], inplace=True)

# Nasarawa State is missing from the PGR projection data, so we will add a row for Nasarawa in this dataframe and fill PGR_BOTH_2022 as 2.8 and PGR_PROJ_2025 as (1+(PGR_BOTH_2022/100))^3
# Source: https://citypopulation.de/en/nigeria/admin/16__nasarawa/
nasarawa_row = pd.DataFrame({
    'ADM1_NAME': ['Nasarawa'],
    'PGR_BOTH_2022': [2.8]
})
pgr = pd.concat([pgr, nasarawa_row], ignore_index=True)

# Calculate PGR_PROJ_2025 for all states using the formula: (1 + (PGR_BOTH_2022 / 100))^3
pgr['PGR_PROJ_2025'] = (1 + (pgr['PGR_BOTH_2022'] / 100)) ** 3

print(pop.columns)
print(pgr.columns)

  ADM0_NAME ADM1_NAME          ADM2_NAME    SenDist_en  POP_2020
0   NIGERIA      ABIA          ABA NORTH    Abia South    115305
1   NIGERIA      ABIA          ABA SOUTH    Abia South    384940
2   NIGERIA      ABIA          AROCHUKWU    Abia North    255439
3   NIGERIA      ABIA              BENDE    Abia North    244495
4   NIGERIA      ABIA            IKWUANO  Abia Central    316803
5   NIGERIA      ABIA  Isiala-Ngwa North  Abia Central    221635
6   NIGERIA      ABIA  Isiala-Ngwa South  Abia Central    165135
7   NIGERIA      ABIA         Isiukwuato    Abia North    181063
8   NIGERIA      ABIA           Obi Ngwa    Abia South    238027
9   NIGERIA      ABIA             OHAFIA    Abia North    353882
109
771
Index(['ADM0_NAME', 'ADM1_NAME', 'ADM2_NAME', 'SenDist_en', 'POP_2020'], dtype='object')
Index(['ADM1_NAME', 'PGR_BOTH_2022', 'PGR_PROJ_2025'], dtype='object')


In [87]:
"""
I will merge the population data with the PGR projection data. The PGR data is at the state level and the population data is at the LGA level.

First, I will need to ensure that the state names in both datasets match.
"""

# Check unique state names in population data
print("Unique state names in population data:")
print(pop['ADM1_NAME'].unique())

# Check unique state names in PGR data
print("\nUnique state names in PGR data:")
print(pgr['ADM1_NAME'].unique())

# It looks like the state names match in both datasets, just that state names in 'pop' are in uppercase. I will convert the state names in 'pgr' to uppercase to ensure they match.
pgr['ADM1_NAME'] = pgr['ADM1_NAME'].str.upper()

# I will drop the row with 'TOTAL' in the PGR data as it is not needed for the merge.
pgr = pgr[pgr['ADM1_NAME'] != 'TOTAL']

# Merge the population data with the PGR projection data on the state name (ADM1_NAME)
pop_pgr = pd.merge(pop, pgr, on='ADM1_NAME', how='left')


Unique state names in population data:
['ABIA' 'ADAMAWA' 'AKWA IBOM' 'ANAMBRA' 'BAUCHI' 'BAYELSA' 'BENUE' 'BORNO'
 'CROSS RIVER' 'DELTA' 'EBONYI' 'EDO' 'EKITI' 'ENUGU'
 'FEDERAL CAPITAL TERRITORY' 'GOMBE' 'IMO' 'JIGAWA' 'KADUNA' 'KANO'
 'KATSINA' 'KEBBI' 'KOGI' 'KWARA' 'LAGOS' 'NASARAWA' 'NIGER' 'OGUN' 'ONDO'
 'OSUN' 'OYO' 'PLATEAU' 'RIVERS' 'SOKOTO' 'TARABA' 'YOBE' 'ZAMFARA']

Unique state names in PGR data:
['ABIA' 'ADAMAWA' 'AKWA IBOM' 'ANAMBRA' 'BAUCHI' 'BAYELSA' 'BENUE' 'BORNO'
 'CROSS RIVER' 'DELTA' 'EBONYI' 'EDO' 'EKITI' 'ENUGU'
 'FEDERAL CAPITAL TERRITORY' 'GOMBE' 'IMO' 'JIGAWA' 'KADUNA' 'KANO'
 'KATSINA' 'KEBBI' 'KOGI' 'KWARA' 'LAGOS' 'NIGER' 'OGUN' 'ONDO' 'OSUN'
 'OYO' 'PLATEAU' 'RIVERS' 'SOKOTO' 'TARABA' 'YOBE' 'ZAMFARA' 'NASARAWA'
 'Nasarawa']


In [88]:
print(pop_pgr.head(10))

  ADM0_NAME ADM1_NAME          ADM2_NAME    SenDist_en  POP_2020  \
0   NIGERIA      ABIA          ABA NORTH    Abia South    115305   
1   NIGERIA      ABIA          ABA SOUTH    Abia South    384940   
2   NIGERIA      ABIA          AROCHUKWU    Abia North    255439   
3   NIGERIA      ABIA              BENDE    Abia North    244495   
4   NIGERIA      ABIA            IKWUANO  Abia Central    316803   
5   NIGERIA      ABIA  Isiala-Ngwa North  Abia Central    221635   
6   NIGERIA      ABIA  Isiala-Ngwa South  Abia Central    165135   
7   NIGERIA      ABIA         Isiukwuato    Abia North    181063   
8   NIGERIA      ABIA           Obi Ngwa    Abia South    238027   
9   NIGERIA      ABIA             OHAFIA    Abia North    353882   

   PGR_BOTH_2022  PGR_PROJ_2025  
0           2.35        1.07217  
1           2.35        1.07217  
2           2.35        1.07217  
3           2.35        1.07217  
4           2.35        1.07217  
5           2.35        1.07217  
6           2

In [89]:
pop_pgr['POP_PROJ_2025'] = pop_pgr['POP_2020'] * pop_pgr['PGR_PROJ_2025']
print(pop_pgr[['ADM1_NAME', 'ADM2_NAME', 'POP_2020', 'POP_PROJ_2025']].head(10))

pop_pgr.to_excel('NGA_LGA_Population_PGR_Projections.xlsx', index=False)

  ADM1_NAME          ADM2_NAME  POP_2020  POP_PROJ_2025
0      ABIA          ABA NORTH    115305  123626.530473
1      ABIA          ABA SOUTH    384940  412721.015048
2      ABIA          AROCHUKWU    255439  273873.963119
3      ABIA              BENDE    244495  262140.137617
4      ABIA            IKWUANO    316803  339666.586300
5      ABIA  Isiala-Ngwa North    221635  237630.337638
6      ABIA  Isiala-Ngwa South    165135  177052.748013
7      ABIA         Isiukwuato    181063  194130.267438
8      ABIA           Obi Ngwa    238027  255205.343817
9      ABIA             OHAFIA    353882  379421.567640
